## Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mikhailklemin/kinopoisks-movies-reviews")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

base = Path(path) / "dataset"
label_map = {"neg": 0, "neu": 1, "pos": 2}

def read_one(args):
    fp, cls = args
    text = fp.read_text(encoding="utf-8", errors="replace").strip()
    return {"text": text, "label": cls, "label_id": label_map[cls]}

tasks = [(fp, cls) for cls in label_map for fp in (base / cls).glob("*.txt")]

with ThreadPoolExecutor(max_workers=16) as ex:
    rows = list(ex.map(read_one, tasks))

df = pd.DataFrame(rows)

In [ ]:
df.head()

## Cleaning

In [ ]:
before = len(df)
df = df[df["text"].str.len() > 0]              # blank texts
df = df.drop_duplicates(subset="text")         # duplicate texts
df = df.reset_index(drop=True)
print(f"Removed {before - len(df)} blank/duplicate rows; {len(df)} remain")
print(df["label"].value_counts())

## RuBERT token-length distribution
### Token lengths determine max_length, which affects batch size, training speed, memory use, inference efficiency, and the serving artifact.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

# DeepPavlov/rubert-base-cased supports at most 512 tokens 
# Token length of every text
# add_special_tokens=True includes the [CLS] and [SEP] tokens used during training
lengths = df["text"].apply(
    lambda t: len(tokenizer.encode(t, add_special_tokens=True, truncation=False))
)

print(lengths.describe(percentiles=[.5, .75, .9, .95, .99]))
print("maximum:", lengths.max())
print("share above 512 tokens:", (lengths > 512).mean().round(3))
print("share above 256 tokens:", (lengths > 256).mean().round(3))

### Many reviews exceed 512 tokens. Preserve the beginning and end of long reviews and remove the middle.

In [ ]:
def encode_head_tail(text: str, tokenizer, max_length: int = 512,
                     head: int = 256, tail: int = 254) -> dict:
    """Encode the beginning and end of a long review."""

    ids = tokenizer.encode(text, add_special_tokens=False)

    if len(ids) > max_length - 2:
        ids = ids[:head] + ids[-tail:]

        text = tokenizer.decode(ids, skip_special_tokens=True)



    return tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        return_token_type_ids=True,
    )

## Stratified split: 70% train / 15% validation / 15% test

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=42,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=42,
) 

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(part)} | class shares:\n{part['label'].value_counts(normalize=True).round(3).to_dict()}")

## label_map.json artifact

In [ ]:
import json

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)